# `preprocess_and_split.py` — Data Cleaning & Train/Val/Test Split

## Purpose
Processes the machine-translated English reviews from `data/raw/full_data_en.csv` and produces clean, stratified train/val/test splits in `data/splits/`.

## Processing Pipeline
```
1. Load translated CSV
2. Deep text cleaning:
   a. Strip HTML tags & entities (e.g. &amp; → &)
   b. Remove URLs and emails
   c. Detect and remove garbled/keyboard-spam text
   d. Normalise Vietnamese translation artifacts (filler phrases, repeated punctuation)
   e. Collapse whitespace
3. Normalise aspect label values → {positive, neutral, negative}
4. Drop rows with no text or all-NaN aspect labels
5. Stratified split: 70% train / 15% val / 15% test
6. Document class imbalance to guide loss function choices
```

## Output Files
| File | Description |
|------|-------------|
| `data/splits/train.csv` | Clean training set (70%) |
| `data/splits/val.csv` | Clean validation set (15%) |
| `data/splits/test.csv` | Clean test set (15%) |
| `data/splits/class_distribution.json` | Per-aspect class counts for each split |
| `data/splits/preprocessing_report.md` | Summary of cleaning steps and data statistics |

In [ ]:
import re, unicodedata, html
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from collections import Counter
import os, json, logging

logging.basicConfig(level=logging.INFO, format='%(asctime)s [%(levelname)s] %(message)s', datefmt='%Y-%m-%d %H:%M:%S')
log = logging.getLogger(__name__)

## Configuration

In [ ]:
import os
ML_RESEARCH = os.path.dirname(os.path.abspath(''))

DATA_PATH      = os.path.join(ML_RESEARCH, 'data', 'raw', 'full_data_en.csv')  # Translated English reviews
OUTPUT_DIR     = os.path.join(ML_RESEARCH, 'data', 'splits')                 # Where train/val/test CSVs are saved
TRAIN_SPLIT    = 0.70                          # 70% training
VAL_SPLIT      = 0.15                          # 15% validation
TEST_SPLIT     = 0.15                          # 15% test
RANDOM_SEED    = 42                            # Reproducibility
TEXT_COLUMNS   = ['data']                      # Column(s) to clean
ASPECT_COLUMNS = ['stayingpower', 'texture', 'smell', 'price', 'colour', 'shipping', 'packing']

# ── Garbled text detection thresholds ──────────────────────────────────────────
GARBLED_MIN_LENGTH      = 6     # Only inspect tokens of at least this length
GARBLED_CONSONANT_RATIO = 0.82  # > 82% consonants → garbled (e.g. 'kslfmwq')
GARBLED_REPEAT_RATIO    = 0.60  # Single character > 60% of token → repeated spam
SPAM_TOKEN_RATIO        = 0.40  # > 40% of tokens garbled → drop whole sentence

## Text Cleaning Functions

These functions form a **pipeline** applied in order to each review text. They handle the common noise patterns seen in machine-translated Vietnamese reviews.

In [ ]:
def clean_html(text: str) -> str:
    """Remove HTML tags and decode HTML entities (e.g. &amp; → &, &#39; → ')."""
    text = html.unescape(text)           # Decode &amp; &lt; &gt; etc.
    text = re.sub(r'<[^>]+>', ' ', text) # Strip remaining <tag> elements
    return text


def clean_urls_and_emails(text: str) -> str:
    """Remove URLs and email addresses — they carry no sentiment signal."""
    text = re.sub(r'http\S+|www\.\S+', ' ', text)      # Remove URLs
    text = re.sub(r'\S+@\S+\.\S+',    ' ', text)       # Remove emails
    return text


def is_garbled_token(token: str) -> bool:
    """
    Heuristic to detect keyboard-spam tokens like 'kslfmwq' or 'aaaaaaa'.
    A token is garbled if:
      - Length >= GARBLED_MIN_LENGTH (short tokens are usually real words)
      - More than GARBLED_CONSONANT_RATIO of its characters are consonants, OR
      - A single character makes up more than GARBLED_REPEAT_RATIO of the token
    """
    if len(token) < GARBLED_MIN_LENGTH: return False
    consonants = set('bcdfghjklmnpqrstvwxyz')
    letters    = [c.lower() for c in token if c.isalpha()]
    if not letters: return False
    cons_ratio   = sum(1 for c in letters if c in consonants) / len(letters)
    repeat_ratio = max(Counter(letters).values()) / len(letters)
    return cons_ratio > GARBLED_CONSONANT_RATIO or repeat_ratio > GARBLED_REPEAT_RATIO


def remove_garbled_sentences(text: str) -> str:
    """
    Remove sentences where more than SPAM_TOKEN_RATIO of tokens are garbled.
    Operates at sentence level (split on . ! ?) to preserve the valid parts of a review.
    """
    sentences = re.split(r'(?<=[.!?])\s+', text)
    clean_sentences = []
    for sent in sentences:
        tokens       = sent.split()
        if not tokens: continue
        garbled_frac = sum(1 for t in tokens if is_garbled_token(t)) / len(tokens)
        if garbled_frac <= SPAM_TOKEN_RATIO:
            clean_sentences.append(sent)   # Keep sentence if mostly non-garbled
    return ' '.join(clean_sentences)


# Common patterns left behind by vi→en machine translation
TRANSLATION_ARTIFACTS = [
    r'\bthe product is\b', r'\bthe goods\b', r'\bgoods received\b',
    r'\.{3,}',   # Excessive ellipsis → single period
    r'!{2,}',    # Multiple exclamation marks → single
    r'\?{2,}',   # Multiple question marks → single
]

def normalize_artifacts(text: str) -> str:
    """Remove/fix common translation artifacts and normalise punctuation."""
    for pattern in TRANSLATION_ARTIFACTS:
        text = re.sub(pattern, ' ', text, flags=re.IGNORECASE)
    text = re.sub(r'\s+', ' ', text)  # Collapse multiple whitespace into single space
    return text.strip()


def normalize_unicode(text: str) -> str:
    """Normalise unicode to NFC form and strip zero-width characters."""
    text = unicodedata.normalize('NFC', text)                  # Canonical composition
    text = re.sub(r'[\u200b-\u200f\u202a-\u202e]', '', text)  # Remove zero-width chars
    return text


def clean_text(text: str) -> str:
    """Full cleaning pipeline: HTML → URLs → garbled → artifacts → unicode."""
    text = clean_html(text)
    text = clean_urls_and_emails(text)
    text = remove_garbled_sentences(text)
    text = normalize_artifacts(text)
    text = normalize_unicode(text)
    return text.strip()

## Label Normalisation
Some rows use inconsistent label values (e.g. '1' instead of 'positive', capitalised variants). This maps everything to the canonical set.

In [ ]:
LABEL_SYNONYMS = {
    'positive': 'positive', 'pos': 'positive', '1': 'positive',
    'neutral':  'neutral',  'neu': 'neutral',  '0': 'neutral',
    'negative': 'negative', 'neg': 'negative', '-1': 'negative',
}

def normalize_labels(df: pd.DataFrame) -> pd.DataFrame:
    """Map all aspect labels to their canonical form (positive/neutral/negative)."""
    for asp in ASPECT_COLUMNS:
        if asp not in df.columns: continue
        df[asp] = df[asp].apply(
            lambda v: LABEL_SYNONYMS.get(str(v).lower().strip()) if pd.notna(v) else None
        )
    return df

## Stratified Splitting
Uses sklearn's `train_test_split` with `stratify` to maintain the same class ratios across all splits. Stratification is done on a combined sentiment string (e.g. `'pos|neg|pos|...'`) to respect the multi-aspect label structure.

In [ ]:
def stratified_split(df: pd.DataFrame):
    """
    Performs a stratified 70/15/15 split.
    The stratification key is a concatenation of all aspect labels per row.
    Rows with unique label patterns are put into training to avoid empty strata.
    """
    # Build a combined label string used as stratification key
    df['_strat_key'] = df[ASPECT_COLUMNS].fillna('na').apply(
        lambda row: '|'.join(str(v)[:3] for v in row), axis=1  # e.g. 'pos|neg|pos|neu|pos|na|na'
    )

    # Rows with categories appearing only once can't be stratified → put in train
    freq          = df['_strat_key'].value_counts()
    rare_mask     = df['_strat_key'].isin(freq[freq < 2].index)
    rare_df       = df[rare_mask]
    stratified_df = df[~rare_mask]

    # First split: 70% train vs 30% remainder
    train, remainder = train_test_split(
        stratified_df, test_size=(VAL_SPLIT + TEST_SPLIT),
        random_state=RANDOM_SEED, stratify=stratified_df['_strat_key']
    )
    # Second split: 50/50 of the 30% → 15% val, 15% test
    val, test = train_test_split(
        remainder, test_size=0.5,
        random_state=RANDOM_SEED, stratify=remainder['_strat_key']
    )

    # Add rare rows to training only
    train = pd.concat([train, rare_df], ignore_index=True)

    # Clean up the temporary key column
    for part in [train, val, test]:
        part.drop(columns=['_strat_key'], inplace=True)

    return train, val, test


def main():
    os.makedirs(OUTPUT_DIR, exist_ok=True)

    log.info('Loading data from %s', DATA_PATH)
    df = pd.read_csv(DATA_PATH)
    log.info('Loaded %d rows', len(df))

    # Step 1: Clean text
    log.info('Applying text cleaning pipeline...')
    for col in TEXT_COLUMNS:
        if col in df.columns:
            df[col] = df[col].astype(str).apply(clean_text)

    # Step 2: Drop rows with empty text after cleaning
    df = df[df['data'].str.strip().astype(bool)]

    # Step 3: Normalise labels
    df = normalize_labels(df)

    # Step 4: Drop rows with no aspect labels at all
    df = df.dropna(subset=ASPECT_COLUMNS, how='all')

    log.info('Clean dataset: %d rows', len(df))

    # Step 5: Stratified split
    train, val, test = stratified_split(df)

    # Step 6: Save splits
    train.to_csv(f'{OUTPUT_DIR}/train.csv', index=False)
    val.to_csv(  f'{OUTPUT_DIR}/val.csv',   index=False)
    test.to_csv( f'{OUTPUT_DIR}/test.csv',  index=False)

    log.info('Split: train=%d  val=%d  test=%d', len(train), len(val), len(test))


main()